In [ ]:
import pandas as pd

from dap_job_quality import BUCKET_NAME
from dap_job_quality.getters import ojo_getters as ojo
from dap_job_quality.getters.keywords import get_keywords
from dap_job_quality.utils import analysis_utils

In [ ]:
def prop_table(full_data, grouping_cols = ['knowledge_domain','sector'], prop_col='FLEX_HOURS', sort_by = ['proportion'], ascending=False):
    prop_table = full_data.groupby(grouping_cols).agg({prop_col: sum, 'id': 'size'})
    prop_table[f'non_{prop_col}'] = prop_table['id'] - prop_table[prop_col]
    prop_table['proportion'] = prop_table[prop_col] / prop_table['id']
    return prop_table.sort_values(sort_by, ascending=ascending)

In [ ]:
processed_data = pd.read_parquet('s3://open-jobs-lake/job_quality/outputs/random/job_ads_prod_True_n_100000.parquet')

In [ ]:
titles = ojo.get_ojo_job_title_sample()
locations = ojo.get_ojo_location_sample()
salaries = ojo.get_ojo_salaries_sample()
skills = ojo.get_ojo_skills_sample()
lookup = get_keywords()

In [ ]:
processed_data = pd.merge(processed_data, lookup[['target_phrase', 'subcategory','dimension']], on='target_phrase', how='left')
processed_data.head()

In [ ]:
dimensions_wide = analysis_utils.create_wide_table(processed_data)
dimensions_wide

In [ ]:
full_data = pd.merge(titles, dimensions_wide, on='id', how='left')
full_data = pd.merge(locations, full_data, on='id', how='left')

In [ ]:
columns_to_replace = dimensions_wide.columns[1:] # the first column is the id
# These columns have NaN where there are *no* mentions of JQ dimensions in these job adverts
full_data[columns_to_replace] = full_data[columns_to_replace].fillna(0)
full_data.head()

In [ ]:
prop_flex_hours_by_kd = prop_table(full_data, grouping_cols = ['knowledge_domain'], prop_col='FLEX_HOURS')
prop_flex_hours_by_kd

In [ ]:
prop_flex_hours_by_sector = prop_table(full_data, grouping_cols = ['sector'], prop_col='FLEX_HOURS')
prop_flex_hours_by_sector

In [ ]:
prop_flex_hours_by_kd.to_csv('outputs/prop_flex_hours_by_kd.csv')

In [ ]:
full_data['created']

In [ ]:
full_data['created'] = pd.to_datetime(full_data['created'])
full_data['year'] = full_data['created'].dt.year

In [ ]:
prop_flex_loc_by_kd = prop_table(full_data, grouping_cols = ['knowledge_domain', 'year'], prop_col='FLEX_LOC', sort_by=['knowledge_domain', 'year'], ascending=True)
prop_flex_loc_by_kd

In [ ]:
prop_flex_loc_by_kd.to_csv('outputs/prop_flex_loc_by_kd.csv')